# 3 &middot; Hochschild (co)homology

For a finite-dimensional algebra $A$, Hochschild cohomology $HH^\bullet(A)$ and
homology $HH_\bullet(A)$ are computed here from the **normalized bar complex**.
The bar complex is exponentially large &mdash; the degree-$n$ cochains of a
$d$-dimensional algebra number $d(d-1)^n$ &mdash; so it is an honest *oracle* for
small algebras only. But every entry it returns is an **exact rank**: no
rounding, and the answer is characteristic-correct.

The entry points are `A.hochschild_cohomology(top)` and
`A.hochschild_homology(top)`, each returning an `HHTable`.

In [1]:
from quiverlab import CC, GF, truncated_polynomial, linear_path_algebra

## $k[x]/(x^n)$: the characteristic matters

Take $A = k[x]/(x^3)$. Over a field whose characteristic does **not** divide
$n=3$, the Hochschild cohomology settles into a steady pattern.

In [2]:
cc = truncated_polynomial(3, field=CC).hochschild_cohomology(4)
print(cc)

HH^n dimensions for Algebra of dimension 3 over CC (computing exactly in QQ) (engine: normalized bar complex)
HH^0 = 3  HH^1 = 2  HH^2 = 2  HH^3 = 2  HH^4 = 2


Reading it: $HH^0(A)=Z(A)=A$ is the whole $3$-dimensional (commutative)
algebra. For $i\ge 1$ the dimensions stabilize at $2$. This is the generic
picture in characteristic $0$ (and in any characteristic prime to $n$).

Now change **one word** &mdash; the field &mdash; to $\mathbb{F}_3$, where
$3\mid n$:

In [3]:
gf3 = truncated_polynomial(3, field=GF(3)).hochschild_cohomology(4)
print(gf3)

HH^n dimensions for Algebra of dimension 3 over GF(3) (engine: normalized bar complex)
HH^0 = 3  HH^1 = 3  HH^2 = 3  HH^3 = 3  HH^4 = 3


## The $\operatorname{char}\mid n$ jump

Over $\mathbb{F}_3$ the dimensions are $[3,3,3,3,3]$ instead of
$[3,2,2,2,\dots]$: every $HH^i$ with $i\ge1$ gains a dimension. The mechanism is
classical &mdash; the connecting maps in the resolution of $k[x]/(x^n)$ carry
factors of $n$ (think $x\mapsto n x^{n-1}$), and when $\operatorname{char}k\mid n$
those maps vanish, so nothing cancels. Same algebra, same presentation; only the
field changed. This is why quiverlab treats the ground field as data, not
decoration.

For contrast, $\mathbb{F}_2$ (where $2\nmid 3$) agrees with the
characteristic-$0$ answer:

In [4]:
gf2 = truncated_polynomial(3, field=GF(2)).hochschild_cohomology(4)
print("HH^* over GF(2):", gf2.dims, " (2 does not divide 3 -> same as CC)")

HH^* over GF(2): [3, 2, 2, 2, 2]  (2 does not divide 3 -> same as CC)


## Hereditary algebras: cohomology in low degree only

If $A$ is *hereditary* (global dimension $\le 1$) &mdash; for instance the path
algebra of an acyclic quiver &mdash; then $HH^i(A)=0$ for all $i\ge 2$. Take the
linear quiver $1\to2\to3$:

In [5]:
her = linear_path_algebra(3, field=CC).hochschild_cohomology(2)
print(her)

HH^n dimensions for Algebra of dimension 6 over CC (computing exactly in QQ) (engine: normalized bar complex)
HH^0 = 1  HH^1 = 0  HH^2 = 0


$HH^0=k$ because the algebra is connected (its center is just the scalars),
$HH^1=0$ because the quiver is a tree (no oriented cycles, hence no outer
derivations), and everything from degree $2$ up vanishes with the global
dimension. Compare the truncated polynomial algebra above, whose cohomology
never stops &mdash; it has infinite global dimension.

## Homology vs. cohomology; symmetric algebras

$k[x]/(x^n)$ is a commutative Frobenius, hence **symmetric**, algebra. For a
symmetric algebra there is a duality making $HH_i(A)\cong HH^i(A)$. We can watch
the two sequences coincide:

In [6]:
A = truncated_polynomial(3, field=CC)
co = A.hochschild_cohomology(4)
ho = A.hochschild_homology(4)
print("HH^* :", co.dims)
print("HH_* :", ho.dims)
print("same dimensions in every degree?", co.dims == ho.dims)

HH^* : [3, 2, 2, 2, 2]
HH_* : [3, 2, 2, 2, 2]
same dimensions in every degree? True


The two sequences agree degree by degree &mdash; the symmetric-algebra
duality, made visible. It is not an artifact of characteristic $0$; over
$\mathbb{F}_3$ homology and cohomology *both* become $[3,3,3,3,3]$, still equal:

In [7]:
B = truncated_polynomial(3, field=GF(3))
print("HH^* over GF(3):", B.hochschild_cohomology(4).dims)
print("HH_* over GF(3):", B.hochschild_homology(4).dims)

HH^* over GF(3): [3, 3, 3, 3, 3]
HH_* over GF(3): [3, 3, 3, 3, 3]


## Reading an `HHTable`

The returned object is a thin, honest record: a list of dimensions plus enough
metadata to know what you are looking at. It indexes and iterates by degree, and
its `repr` names the engine that produced it.

In [8]:
t = truncated_polynomial(3, field=CC).hochschild_cohomology(4)
print("kind       :", t.kind)          # "HH^" for cohomology, "HH_" for homology
print("top degree :", t.top)
print("dims       :", t.dims)
print("t[2]       :", t[2])            # HH^2
print("list(t)    :", list(t))
print()
print(repr(t))

kind       : HH^
top degree : 4
dims       : [3, 2, 2, 2, 2]
t[2]       : 2
list(t)    : [3, 2, 2, 2, 2]

HH^n dimensions for Algebra of dimension 3 over CC (computing exactly in QQ) (engine: normalized bar complex)
HH^0 = 3  HH^1 = 2  HH^2 = 2  HH^3 = 2  HH^4 = 2


## Honesty about scope

The bar oracle is exponential: a `max_cells` guard raises `DepthLimitError`
rather than quietly grinding through an intractable matrix, so you always know
when you have left the certified range. Deeper, polynomial resolutions
(Bardzell, minimal, Chouhy&ndash;Solotar) are future work. Within range, though,
every number on these tables is an exact rank over an exact field &mdash;
reproducible, and honest about the characteristic it was computed in.